In [1]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [2]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [35]:
import networkx as nx
import numpy as np
import pandas as pd

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,n_unique_chars,word_byte,word_id
0,abdom,abdom,5,abdmo,"{b, m, o, a, d}",5,20491,0
1,abend,abend,5,abden,"{n, b, e, a, d}",5,8219,1
2,abets,abets,5,abest,"{b, e, s, a, t}",5,786451,2
3,abhor,abhor,5,abhor,"{r, b, h, o, a}",5,147587,3
4,abide,abide,5,abdei,"{b, e, i, a, d}",5,283,4


# IMPORT OUTPUT DATA

In [17]:
ifpn = os.path.join(rc.output_folder, 'l5.txt')
l5_df = pd.read_csv(filepath_or_buffer=ifpn, sep = '\t', dtype = np.int32)

In [21]:
l5_df.head()

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327
1,16912387,8914984,1344516,4202944,35668496,25827371,27171887,31374831,67043327
2,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327
3,16912387,8914984,1351744,4195716,35668496,25827371,27179115,31374831,67043327
4,16912387,8914984,35668496,1344516,4202944,25827371,61495867,62840383,67043327


In [22]:
# how many unique letter groups?
for il in range(2, 6):
    cn  = f"l{il}"
    print(cn, l5_df[cn].unique().shape)

l2 (2005,)
l3 (2131,)
l4 (649,)
l5 (11,)


# get words

In [23]:
for idx in range(1, 6):
    cn = f"w{str(idx)}b"
    ncn = f"w{str(idx)}"
    l5_df[ncn] = l5_df[cn].map(word_byte_to_word_dict)

In [24]:
# get unique word combos
output_list = []
for i_row, row in l5_df.iterrows():
    my_set = set()
    for cn_idx in range(1, 6):
        cn = f"w{cn_idx}"
        my_set.add(row[cn])

    output = tuple(sorted(my_set))

    output_list.append(output)

    

In [25]:
l5_df['word_group'] = output_list

In [26]:
l5_df = l5_df.sort_values(by = ['w1', 'w2', 'w3', 'w4', 'w5']).reset_index(drop = True)

In [27]:
l5_df.head()

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5,w1,w2,w3,w4,w5,word_group
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327,ambry,fldxt,pucks,vejoz,whing,"(ambry, fldxt, pucks, vejoz, whing)"
1,16912387,8914984,1344516,4202944,35668496,25827371,27171887,31374831,67043327,ambry,fldxt,pucks,whing,vejoz,"(ambry, fldxt, pucks, vejoz, whing)"
2,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327,ambry,fldxt,pungs,vejoz,whick,"(ambry, fldxt, pungs, vejoz, whick)"
3,16912387,8914984,1351744,4195716,35668496,25827371,27179115,31374831,67043327,ambry,fldxt,pungs,whick,vejoz,"(ambry, fldxt, pungs, vejoz, whick)"
4,16912387,8914984,35668496,1344516,4202944,25827371,61495867,62840383,67043327,ambry,fldxt,vejoz,pucks,whing,"(ambry, fldxt, pucks, vejoz, whing)"


In [28]:
l5_df['word_group_hash'] = l5_df['word_group'].map(hash)

In [29]:
l5_df['word_group_hash'].unique().shape

(538,)

In [30]:
l5_df = l5_df.drop_duplicates(subset = 'word_group_hash').reset_index(drop = True)

In [31]:
ofpn = os.path.join(rc.output_folder, 'l5_output.xlsx')
l5_df.to_excel(excel_writer=ofpn, index = False)

# build a cool graph!

In [ ]:
# build the groups

In [32]:
def build_a_word_tuple(row, level:int):
    cn_list = []
    for ii in range(1, level + 1):
        cn = f"w{ii}"
        cn_list.append(row[cn])

    cn_list = sorted(cn_list)
    return tuple(cn_list)


In [33]:
l5_df['l2_words'] = l5_df.apply(func = build_a_word_tuple, axis = 1, level = 2)
l5_df['l3_words'] = l5_df.apply(func = build_a_word_tuple, axis = 1, level = 3)
l5_df['l4_words'] = l5_df.apply(func = build_a_word_tuple, axis = 1, level = 4)
l5_df['l5_words'] = l5_df.apply(func = build_a_word_tuple, axis = 1, level = 5)

In [34]:
l5_df.head()

,w1b,w2b,w3b,w4b,w5b,l2,l3,l4,l5,w1,w2,w3,w4,w5,word_group,word_group_hash,l2_words,l3_words,l4_words,l5_words
0,16912387,8914984,1344516,35668496,4202944,25827371,27171887,62840383,67043327,ambry,fldxt,pucks,vejoz,whing,"(ambry, fldxt, pucks, vejoz, whing)",-5250238401717725197,"(ambry, fldxt)","(ambry, fldxt, pucks)","(ambry, fldxt, pucks, vejoz)","(ambry, fldxt, pucks, vejoz, whing)"
1,16912387,8914984,1351744,35668496,4195716,25827371,27179115,62847611,67043327,ambry,fldxt,pungs,vejoz,whick,"(ambry, fldxt, pungs, vejoz, whick)",-1085327590955133636,"(ambry, fldxt)","(ambry, fldxt, pungs)","(ambry, fldxt, pungs, vejoz)","(ambry, fldxt, pungs, vejoz, whick)"
2,25202689,850,2121764,458888,35130368,25203539,27325303,27784191,62914559,ampyx,bejig,fconv,hdqrs,klutz,"(ampyx, bejig, fconv, hdqrs, klutz)",5662462088694696788,"(ampyx, bejig)","(ampyx, bejig, fconv)","(ampyx, bejig, fconv, hdqrs)","(ampyx, bejig, fconv, hdqrs, klutz)"
3,25202689,4194642,2121764,458888,35130368,29397331,31519095,31977983,67108351,ampyx,bewig,fconv,hdqrs,klutz,"(ampyx, bewig, fconv, hdqrs, klutz)",-7114037463363095490,"(ampyx, bewig)","(ampyx, bewig, fconv)","(ampyx, bewig, fconv, hdqrs)","(ampyx, bewig, fconv, hdqrs, klutz)"
4,25202689,34226178,6291844,2616,1320000,59428867,65720711,65723327,67043327,ampyx,bortz,chivw,fjeld,gunks,"(ampyx, bortz, chivw, fjeld, gunks)",7264298577767736369,"(ampyx, bortz)","(ampyx, bortz, chivw)","(ampyx, bortz, chivw, fjeld)","(ampyx, bortz, chivw, fjeld, gunks)"


In [36]:
# BUILD THE GRAPH

In [48]:
my_graph = nx.DiGraph()
for i_row, row in l5_df.iterrows():

    # # l2 to l3
    # my_graph.add_edge(row['l2_words'], row['l3_words'], weight = 1)
    

    # # l3 to l4
    # my_graph.add_edge(row['l3_words'], row['l4_words'], weight = 2)

    # # l4 to l5
    # my_graph.add_edge(row['l4_words'], row['l5_words'], weight = 3)

    # l1 to l2
    my_graph.add_edge(row['w1'], row['w2'], weight = 1)
    my_graph.add_edge(row['w2'], row['w3'], weight = 2)
    my_graph.add_edge(row['w3'], row['w4'], weight = 3)
    my_graph.add_edge(row['w4'], row['w5'], weight = 4)


In [49]:
my_graph.number_of_edges()

1131

In [50]:
my_graph.number_of_nodes()

493

In [51]:
# export to gephi
nx.write_gexf(G = my_graph, path = os.path.join(rc.output_folder, 'test_graph.gexf'))

In [ ]:
testo = l5_df[['l2_words', 'l3_words']].to

In [ ]:
testo

In [ ]:
my_graph.add_edges_from(].to_records())